# ch06 Bonus 02：IMDb 影评分类（真实数据集）

> 对照官方 `ch06/03_bonus_imdb-classification`

## 一句话

在 **IMDb 5 万条影评**（2.5 万正 / 2.5 万负）上做情感分类，对比我们微调的 GPT 与其它模型（如 sklearn 的 TF-IDF + SVM）的效果。

## 数据集

IMDb 是经典情感分类基准：
- 5 万条影评，二分类（正面/负面）
- 预划分 train/test 各 2.5 万

> 本 notebook 尝试下载 IMDb；离线/网络受限时**回退到自造数据**继续演示流程（与 ch05 bonus 01 的 Gutenberg 处理方式一致）。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import tiktoken
from src.gpt import GPTModel, GPT_CONFIG_124M

# 尝试加载 IMDb（官方提供的 csv 子集路径），失败则回退自造数据
imdb_path = Path("data/IMDB Dataset.csv")
use_imdb = False
if imdb_path.exists():
    import pandas as pd
    df = pd.read_csv(imdb_path)
    # IMDb csv: 列 review(文本) + sentiment(positive/negative)
    texts = df["review"].tolist()[:1000]   # 取子集 demo（全量 5 万太慢）
    labels = [1 if s == "positive" else 0 for s in df["sentiment"][:1000]]
    use_imdb = True
    print(f"✓ 加载 IMDb: {len(texts)} 条（子集）")
else:
    print(f"未找到 {imdb_path}，回退到自造中文情感数据。")
    print("获取 IMDb: 下载 https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset")
    POS = ["这部电影非常精彩 我很喜欢", "太好看了 剧情感人至深", "画面优美 值得推荐",
           "演技出色 故事动人", "完美之作 强烈推荐", "音乐动听 视觉震撼",
           "节奏紧凑 引人入胜", "结局温暖 回味无穷"]
    NEG = ["太糟糕了 浪费时间", "剧情无聊 让人失望", "画面粗糙 毫无诚意",
           "演技尴尬 故事混乱", "简直烂片 不忍直视", "噪音刺耳 看不下去",
           "节奏拖沓 昏昏欲睡", "结局糟糕 一无是处"]
    texts = POS + NEG
    labels = [1]*len(POS) + [0]*len(NEG)
print(f"使用 {'IMDb' if use_imdb else '自造数据'}: {len(texts)} 条")

In [ ]:
tok = tiktoken.get_encoding("gpt2")

class SentiDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128, pad_id=50256):
        self.max_len, self.pad_id = max_len, pad_id
        self.data = [(tokenizer.encode(t)[:max_len], l) for t, l in zip(texts, labels)]
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        ids, l = self.data[i]
        ids = ids + [self.pad_id]*(self.max_len-len(ids))
        return torch.tensor(ids), torch.tensor(l)

# 划分 train/val
n = len(texts)
n_train = int(n * 0.8)
train_ds = SentiDataset(texts[:n_train], labels[:n_train], tok)
val_ds = SentiDataset(texts[n_train:], labels[n_train:], tok)
train_dl = DataLoader(train_ds, batch_size=8, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=8)
print(f"train {len(train_ds)} / val {len(val_ds)}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = dict(GPT_CONFIG_124M)
cfg.update({"emb_dim":128, "n_layers":2, "n_heads":4, "context_length":128})
torch.manual_seed(123)
model = GPTModel(cfg)
model.out_head = nn.Linear(cfg["emb_dim"], 2)
for p in model.parameters(): p.requires_grad = False
for p in model.trf_blocks[-1].parameters(): p.requires_grad = True
for p in model.final_norm.parameters(): p.requires_grad = True
for p in model.out_head.parameters(): p.requires_grad = True
model.to(device)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=5e-4)

# IMDb 子集训练轮数多些；自造数据少则快
epochs = 3 if use_imdb else 15
model.train()
for ep in range(epochs):
    total = n = 0
    for x, y in train_dl:
        opt.zero_grad()
        loss = F.cross_entropy(model(x.to(device))[:, -1, :], y.to(device))
        loss.backward(); opt.step()
        total += loss.item(); n += 1
    print(f"epoch {ep}: train loss {total/n:.4f}")

In [ ]:
def accuracy(dl):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in dl:
            pred = model(x.to(device))[:, -1, :].argmax(-1)
            correct += (pred==y.to(device)).sum().item(); total += len(y)
    return correct/total

print(f"train acc: {100*accuracy(train_dl):.1f}%")
print(f"val   acc: {100*accuracy(val_dl):.1f}%")
print("\n💡 IMDb 上微调 GPT 通常能达 85%+，超过 sklearn TF-IDF+SVM (~88%) 略低，")
print("   但加载 OpenAI 预训练权重后可达 90%+，超越传统方法。")
print("   （本 demo 用未训练的小模型，仅验证流程。）")